# nb142 — Analogy chain: cross-assay proxy features (pillar 3)

The strategic move. For each PXR compound (train + test):
  1. Find top-k=20 Tanimoto neighbors in the ChEMBL bulk dataset (10M+ records, ~14k targets/assays)
  2. Retrieve their multi-assay activity profile across thousands of assays
  3. Build a query-compound × assay matrix

Then on PXR train side:
  4. Find which assays' neighbor-inferred activities best CORRELATE with the known PXR pEC50
  5. These 'proxy assays' are our analogy chain — they let us 'see' a compound's behavior via its neighbors' behavior

Features for PXR ML model = top-K most-correlated proxy assay values per compound.

In [ ]:
import os, subprocess, sys, glob
os.environ['PYTHONUNBUFFERED'] = '1'
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'rdkit', 'pyarrow', 'tqdm'], check=False)
import pandas as pd, numpy as np
from pathlib import Path

# Load ChEMBL bulk parquet (from nb140)
bulk_path = None
for p in glob.glob('/kaggle/input/**/chembl_bulk_activities.parquet', recursive=True):
    bulk_path = p; break
if not bulk_path:
    for p in glob.glob('/kaggle/input/**/papyrus_pxr_related_filtered.parquet', recursive=True):
        bulk_path = p; break
print(f'Using: {bulk_path}')
df = pd.read_parquet(bulk_path)
print(f'  {len(df):,} rows  cols={list(df.columns)[:15]}')

In [ ]:
# Normalize
smi_col = 'canonical_smiles' if 'canonical_smiles' in df.columns else ('SMILES' if 'SMILES' in df.columns else 'SMILES_Stripped')
val_col = 'pchembl_value' if 'pchembl_value' in df.columns else 'pchembl_value_Mean'
assay_col = 'assay_chembl_id' if 'assay_chembl_id' in df.columns else 'accession'
df = df.dropna(subset=[smi_col, val_col, assay_col])
df = df.rename(columns={smi_col:'smi', val_col:'pchembl', assay_col:'assay'})
df = df[['smi','pchembl','assay']]
# Per (compound, assay) median
df_agg = df.groupby(['smi','assay'])['pchembl'].median().reset_index()
print(f'After aggregation: {len(df_agg):,}  compounds={df_agg["smi"].nunique():,}  assays={df_agg["assay"].nunique():,}')

# Keep only assays with sufficient density (>= 30 compounds tested)
assay_n = df_agg.groupby('assay').size()
good_assays = assay_n[assay_n >= 30].index
df_agg = df_agg[df_agg['assay'].isin(good_assays)].reset_index(drop=True)
print(f'After density filter (>=30 compounds): {len(df_agg):,}  assays={df_agg["assay"].nunique():,}')

In [ ]:
# Build reference compound list + fingerprints
from rdkit import Chem
from rdkit.Chem import AllChem, DataStructs
import time
ref_smiles = df_agg['smi'].unique().tolist()
print(f'Reference compounds: {len(ref_smiles):,}')

def morgan_fp(s):
    m = Chem.MolFromSmiles(s)
    if m is None: return None
    return AllChem.GetMorganFingerprintAsBitVect(m, 2, 2048)

t0 = time.time()
ref_fps = []
ref_valid_smiles = []
for s in ref_smiles:
    fp = morgan_fp(s)
    if fp is not None:
        ref_fps.append(fp)
        ref_valid_smiles.append(s)
print(f'Valid ref fps: {len(ref_fps):,}  ({time.time()-t0:.0f}s)')
smi_to_idx = {s:i for i,s in enumerate(ref_valid_smiles)}

In [ ]:
# Build sparse compound × assay activity matrix
from scipy.sparse import csr_matrix
assay_list = sorted(df_agg['assay'].unique())
assay_to_idx = {a:i for i,a in enumerate(assay_list)}
print(f'Assays: {len(assay_list):,}')

# Build matrix (values only where measured; sparse)
rows, cols, vals = [], [], []
for s, a, v in zip(df_agg['smi'], df_agg['assay'], df_agg['pchembl']):
    if s in smi_to_idx:
        rows.append(smi_to_idx[s])
        cols.append(assay_to_idx[a])
        vals.append(v)
A = csr_matrix((vals, (rows, cols)), shape=(len(ref_valid_smiles), len(assay_list)))
# Also a sparse 'mask' = 1 where measured
M = csr_matrix(([1.0]*len(vals), (rows, cols)), shape=(len(ref_valid_smiles), len(assay_list)))
print(f'Activity matrix: {A.shape}, nnz={A.nnz:,}')
print(f'Density: {A.nnz / (A.shape[0]*A.shape[1]) * 100:.3f}%')

In [ ]:
# Load PXR train + test
import urllib.request
HF = 'https://huggingface.co/datasets/openadmet/pxr-challenge-train-test/resolve/main'
for fn in ['pxr-challenge_TRAIN.csv', 'pxr-challenge_TEST_BLINDED.csv']:
    p = f'/kaggle/working/{fn}'
    if not Path(p).exists():
        urllib.request.urlretrieve(f'{HF}/{fn}', p)
tr = pd.read_csv('/kaggle/working/pxr-challenge_TRAIN.csv')
te = pd.read_csv('/kaggle/working/pxr-challenge_TEST_BLINDED.csv')
print(f'PXR train: {len(tr)}  test: {len(te)}')

In [ ]:
# For each PXR compound, compute Tanimoto-neighbor weighted assay profile
K = 20  # top-k neighbors
MIN_SIM = 0.30

def expand(query_smiles, label):
    n = len(query_smiles)
    n_a = A.shape[1]
    # Output: avg activity per assay (NaN if no neighbor has measured it)
    feat_avg = np.full((n, n_a), np.nan, dtype=np.float32)
    feat_eng = np.zeros((n, n_a), dtype=np.float32)  # neighbor engagement frac
    feat_max_sim = np.zeros(n)
    feat_n_neighbors = np.zeros(n)
    t0 = time.time()
    for i, qs in enumerate(query_smiles):
        qfp = morgan_fp(qs)
        if qfp is None: continue
        sims = np.array(DataStructs.BulkTanimotoSimilarity(qfp, ref_fps), dtype=np.float32)
        feat_max_sim[i] = sims.max()
        # Top-K above MIN_SIM
        top_idx = np.argsort(sims)[::-1][:K]
        top_idx = top_idx[sims[top_idx] >= MIN_SIM]
        feat_n_neighbors[i] = len(top_idx)
        if len(top_idx) == 0: continue
        top_sims = sims[top_idx]
        # For each assay, compute similarity-weighted average of neighbors WITH measurement
        # A[top_idx] is small slice. M[top_idx] for measured mask.
        sub_A = A[top_idx].toarray()  # K' x n_a
        sub_M = M[top_idx].toarray()
        for a in range(n_a):
            mask = sub_M[:, a] > 0
            if mask.sum() > 0:
                w = top_sims[mask]
                feat_avg[i, a] = np.dot(w, sub_A[mask, a]) / w.sum()
                feat_eng[i, a] = mask.sum() / len(top_idx)
        if (i+1) % 100 == 0:
            print(f'  {label}: {i+1}/{n}  ({time.time()-t0:.0f}s)')
    return feat_avg, feat_eng, feat_max_sim, feat_n_neighbors

print('Expanding train compounds...')
tr_avg, tr_eng, tr_msim, tr_nn = expand(tr['SMILES'].tolist(), 'train')
print('Expanding test compounds...')
te_avg, te_eng, te_msim, te_nn = expand(te['SMILES'].tolist(), 'test')

In [ ]:
# Now: WHICH ASSAYS correlate with PXR pEC50? Compute per-assay Spearman
from scipy.stats import spearmanr
y_tr = tr['pEC50'].values
corrs = []
for a in range(tr_avg.shape[1]):
    col = tr_avg[:, a]
    mask = np.isfinite(col)
    if mask.sum() < 100: continue  # need decent coverage
    rho, pval = spearmanr(col[mask], y_tr[mask])
    corrs.append((assay_list[a], rho, pval, mask.sum()))
df_corr = pd.DataFrame(corrs, columns=['assay','rho','pval','n_covered']).sort_values('rho', key=abs, ascending=False)
print(f'Assays with >=100 covered compounds: {len(df_corr):,}')
print(df_corr.head(30))
df_corr.to_csv('/kaggle/working/assay_correlations.csv', index=False)

In [ ]:
# Save full feature matrices and the top-K most-correlated assays as features
out_dir = Path('/kaggle/working/analogy_chain')
out_dir.mkdir(exist_ok=True)
np.save(out_dir / 'tr_avg.npy', tr_avg)
np.save(out_dir / 'te_avg.npy', te_avg)
np.save(out_dir / 'tr_eng.npy', tr_eng)
np.save(out_dir / 'te_eng.npy', te_eng)
np.save(out_dir / 'tr_msim.npy', tr_msim)
np.save(out_dir / 'te_msim.npy', te_msim)
np.save(out_dir / 'tr_nn.npy', tr_nn)
np.save(out_dir / 'te_nn.npy', te_nn)
with open(out_dir / 'assay_list.txt','w') as f:
    f.write('\n'.join(assay_list))
print(f'Saved analogy chain features ({tr_avg.shape[0]} train × {tr_avg.shape[1]} assays)')